# 02 — Known Binary Cross-Match & Novel Candidates

Evaluate CNN and RF classifier performance against known spectroscopic binary
catalogs (SB9, Gaia DR3 NSS), then identify novel binary candidates that are
not present in any existing catalog. This notebook produces the final candidate
catalog for the project.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.table import Table
from pathlib import Path

# Path setup
PROJECT_ROOT = Path(os.getcwd()).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import crossmatch_catalogs, save_figure
from config import VALIDATION_CONFIG, VIS_CONFIG

%matplotlib inline
plt.rcParams["figure.dpi"] = VIS_CONFIG["dpi"]

print(f"Project root: {PROJECT_ROOT}")
print(f"SB9 catalog:  {VALIDATION_CONFIG['sb9_catalog_path']}")
print(f"Gaia NSS:     {VALIDATION_CONFIG['gaia_nss_table']}")

## 1. Load All Results

In [ ]:
# Load labeled sample and crossmatch validation from label generation
labeled = Table.read(PROJECT_ROOT / "data" / "labeled_sample.fits")
crossmatch_val = Table.read(PROJECT_ROOT / "data" / "crossmatch_validation.fits")

# Load CNN and RF test predictions
cnn_preds = np.load(PROJECT_ROOT / "data" / "cnn_test_preds.npz")
rf_preds = np.load(PROJECT_ROOT / "data" / "rf_test_preds.npz")

cnn_probs = cnn_preds["probabilities"]
rf_probs = rf_preds["probabilities"]
test_indices = cnn_preds["test_indices"]

# Build a merged DataFrame for the test set
test_sample = labeled[test_indices]
df = test_sample.to_pandas()
df["cnn_prob"] = cnn_probs
df["rf_prob"] = rf_probs

# Add crossmatch flags (SB9, Gaia NSS) from the crossmatch validation table
# Match on APOGEE_ID
xmatch_df = crossmatch_val.to_pandas()
if "APOGEE_ID" in xmatch_df.columns:
    xmatch_cols = ["APOGEE_ID"]
    for col in ["sb9_match", "gaia_nss_match"]:
        if col in xmatch_df.columns:
            xmatch_cols.append(col)
    df = df.merge(xmatch_df[xmatch_cols], on="APOGEE_ID", how="left")
    df["sb9_match"] = df.get("sb9_match", pd.Series(False)).fillna(False).astype(bool)
    df["gaia_nss_match"] = df.get("gaia_nss_match", pd.Series(False)).fillna(False).astype(bool)

print(f"Test set size:       {len(df)}")
print(f"SB9 matches:         {df['sb9_match'].sum()}")
print(f"Gaia NSS matches:    {df['gaia_nss_match'].sum()}")
print(f"Known binaries:      {(df['sb9_match'] | df['gaia_nss_match']).sum()}")

## 2. CNN Performance on Known Binaries

How well does the CNN detect stars that are already known spectroscopic binaries
in SB9 and Gaia DR3 NSS?

In [ ]:
DETECTION_THRESHOLD = 0.5

# Recall on SB9
sb9 = df[df["sb9_match"]]
cnn_recall_sb9 = (sb9["cnn_prob"] >= DETECTION_THRESHOLD).mean() if len(sb9) > 0 else np.nan
rf_recall_sb9 = (sb9["rf_prob"] >= DETECTION_THRESHOLD).mean() if len(sb9) > 0 else np.nan

# Recall on Gaia NSS
gaia = df[df["gaia_nss_match"]]
cnn_recall_gaia = (gaia["cnn_prob"] >= DETECTION_THRESHOLD).mean() if len(gaia) > 0 else np.nan
rf_recall_gaia = (gaia["rf_prob"] >= DETECTION_THRESHOLD).mean() if len(gaia) > 0 else np.nan

# Combined known binaries
known = df[df["sb9_match"] | df["gaia_nss_match"]]
cnn_recall_all = (known["cnn_prob"] >= DETECTION_THRESHOLD).mean() if len(known) > 0 else np.nan
rf_recall_all = (known["rf_prob"] >= DETECTION_THRESHOLD).mean() if len(known) > 0 else np.nan

recall_table = pd.DataFrame([
    {"Catalog": "SB9", "N_in_test": len(sb9),
     "CNN_recall": f"{cnn_recall_sb9:.3f}", "RF_recall": f"{rf_recall_sb9:.3f}"},
    {"Catalog": "Gaia NSS", "N_in_test": len(gaia),
     "CNN_recall": f"{cnn_recall_gaia:.3f}", "RF_recall": f"{rf_recall_gaia:.3f}"},
    {"Catalog": "Any known", "N_in_test": len(known),
     "CNN_recall": f"{cnn_recall_all:.3f}", "RF_recall": f"{rf_recall_all:.3f}"},
])
display(recall_table)

# Histogram: CNN probability distribution for known binaries vs labeled singles
fig, ax = plt.subplots(figsize=(8, 5))
singles = df[~(df["sb9_match"] | df["gaia_nss_match"])]
ax.hist(singles["cnn_prob"], bins=50, alpha=0.5, density=True, label="Non-catalog stars")
if len(known) > 0:
    ax.hist(known["cnn_prob"], bins=30, alpha=0.7, density=True, label="Known binaries")
ax.axvline(DETECTION_THRESHOLD, color="k", linestyle="--", alpha=0.5, label="Threshold")
ax.set_xlabel("CNN Binary Probability")
ax.set_ylabel("Density")
ax.set_title("CNN Score Distribution: Known Binaries vs Non-Catalog Stars")
ax.legend()
plt.tight_layout()
save_figure(fig, "cnn_known_binary_hist")
plt.show()

## 3. Missed Binaries

Investigate known binaries that the CNN fails to detect. Are these long-period,
low-amplitude, or otherwise difficult systems?

In [ ]:
# Known binaries missed by CNN (prob < 0.5)
missed = df[(df["sb9_match"] | df["gaia_nss_match"]) & (df["cnn_prob"] < DETECTION_THRESHOLD)]
print(f"Known binaries missed by CNN: {len(missed)}")

if len(missed) > 0:
    display_cols = ["APOGEE_ID", "TEFF", "LOGG", "VSCATTER", "NVISITS", "cnn_prob", "rf_prob"]
    available_cols = [c for c in display_cols if c in missed.columns]
    display(missed[available_cols].sort_values("cnn_prob").head(15))

    # Investigate: are these low-amplitude systems?
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    if "VSCATTER" in missed.columns:
        axes[0].hist(missed["VSCATTER"], bins=20, color="salmon", edgecolor="k", alpha=0.8)
        axes[0].set_xlabel("VSCATTER (km/s)")
        axes[0].set_ylabel("Count")
        axes[0].set_title("RV Scatter of Missed Binaries")

    if "NVISITS" in missed.columns:
        axes[1].hist(missed["NVISITS"], bins=20, color="steelblue", edgecolor="k", alpha=0.8)
        axes[1].set_xlabel("NVISITS")
        axes[1].set_ylabel("Count")
        axes[1].set_title("Number of Visits")

    if "TEFF" in missed.columns and "LOGG" in missed.columns:
        axes[2].scatter(missed["TEFF"], missed["LOGG"], c=missed["cnn_prob"],
                        cmap="RdYlGn", edgecolors="k", linewidths=0.5, s=40)
        axes[2].set_xlabel("Teff (K)")
        axes[2].set_ylabel("log g")
        axes[2].invert_xaxis()
        axes[2].invert_yaxis()
        axes[2].set_title("HR Diagram of Missed Binaries")
        plt.colorbar(axes[2].collections[0], ax=axes[2], label="CNN prob")

    plt.tight_layout()
    save_figure(fig, "missed_binaries_properties")
    plt.show()
else:
    print("No missed binaries — CNN detected all known binaries.")

## 4. Novel Binary Candidates

Stars flagged by the CNN with high probability that are **not** in SB9 or Gaia NSS.
These are our novel discoveries from single-epoch spectral classification.

In [ ]:
# Novel candidates: high CNN prob, NOT in any known binary catalog
NOVEL_THRESHOLD = 0.8
novel = df[(df["cnn_prob"] >= NOVEL_THRESHOLD) & ~df["sb9_match"] & ~df["gaia_nss_match"]]
print(f"Novel binary candidates (CNN prob >= {NOVEL_THRESHOLD}): {len(novel)}")

# Property distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

prop_cols = [("TEFF", "Teff (K)", "steelblue"),
             ("LOGG", "log g", "darkorange"),
             ("FE_H", "[Fe/H]", "seagreen"),
             ("VSCATTER", "VSCATTER (km/s)", "indianred")]

for ax, (col, xlabel, color) in zip(axes.ravel(), prop_cols):
    if col in novel.columns:
        ax.hist(novel[col].dropna(), bins=30, color=color, edgecolor="k", alpha=0.8)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Count")
        ax.set_title(f"Novel Candidates: {xlabel}")
    else:
        ax.set_visible(False)

plt.suptitle(f"Property Distributions of {len(novel)} Novel Binary Candidates", fontsize=13, y=1.01)
plt.tight_layout()
save_figure(fig, "novel_candidates_properties")
plt.show()

# Save novel candidate list
novel_outpath = PROJECT_ROOT / "data" / "novel_binary_candidates.csv"
novel.to_csv(novel_outpath, index=False)
print(f"Saved novel candidates to {novel_outpath}")

## 5. Final Candidate Catalog

In [ ]:
# Build final candidate catalog
# All stars above the detection threshold
candidates = df[df["cnn_prob"] >= DETECTION_THRESHOLD].copy()

# Add RV validation classification from notebook 01
rv_val = pd.read_csv(PROJECT_ROOT / "data" / "rv_validation_results.csv")
if "APOGEE_ID" in rv_val.columns and "classification" in rv_val.columns:
    rv_class_map = rv_val.set_index("APOGEE_ID")["classification"].to_dict()
    candidates["rv_validation_class"] = candidates["APOGEE_ID"].map(rv_class_map).fillna("not_tested")
else:
    candidates["rv_validation_class"] = "not_tested"

# Flag novel candidates
candidates["is_novel"] = ~candidates["sb9_match"] & ~candidates["gaia_nss_match"]

# Select output columns
output_cols = ["APOGEE_ID"]
for col in ["RA", "DEC", "TEFF", "LOGG", "FE_H", "VSCATTER", "NVISITS"]:
    if col in candidates.columns:
        output_cols.append(col)
output_cols += ["cnn_prob", "rf_prob", "sb9_match", "gaia_nss_match",
                "rv_validation_class", "is_novel"]

catalog = candidates[output_cols].sort_values("cnn_prob", ascending=False).reset_index(drop=True)

# Save
catalog_path = PROJECT_ROOT / "data" / "binary_candidates_final.csv"
catalog.to_csv(catalog_path, index=False)
print(f"Final catalog: {len(catalog)} candidates saved to {catalog_path}")
print()
print("Top 20 candidates:")
display(catalog.head(20))

## 6. Summary Statistics

In [ ]:
# Final summary
n_total = len(catalog)
n_sb9 = catalog["sb9_match"].sum()
n_gaia = catalog["gaia_nss_match"].sum()
n_novel = catalog["is_novel"].sum()
n_rv_confirmed = (catalog["rv_validation_class"] == "confirmed").sum()
n_rv_likely = (catalog["rv_validation_class"] == "likely").sum()
n_rv_tested = (catalog["rv_validation_class"] != "not_tested").sum()

print("=" * 60)
print("BINARY STAR DETECTION — FINAL SUMMARY")
print("=" * 60)
print(f"Total candidates (CNN prob >= {DETECTION_THRESHOLD}):  {n_total}")
print(f"  In SB9 catalog:                        {n_sb9}")
print(f"  In Gaia DR3 NSS:                       {n_gaia}")
print(f"  Novel (not in any catalog):             {n_novel}")
print(f"  RV-tested:                              {n_rv_tested}")
print(f"    Confirmed by RV orbit:                {n_rv_confirmed}")
print(f"    Likely (significant period):           {n_rv_likely}")
print("=" * 60)
print()
print(f"We identify {n_novel} novel binary candidates from single-epoch")
print(f"spectra that are not in existing spectroscopic binary catalogs.")